In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import seaborn as seaborn
from collections import Counter
from tensorflow.keras.models import load_model

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("harishkumardatalab/food-image-classification-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/nikita/.cache/kagglehub/datasets/harishkumardatalab/food-image-classification-dataset/versions/1


In [5]:
import tensorflow as tf

print("Версия TensorFlow:", tf.__version__)
print("Доступные GPU:", tf.config.list_physical_devices('GPU'))
print("Устройство для вычислений:", tf.test.is_gpu_available())

Версия TensorFlow: 2.14.0
Доступные GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Устройство для вычислений: True


2026-03-02 22:51:34.394068: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-02 22:51:34.395488: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-02 22:51:34.396670: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

In [6]:
import os
from pathlib import Path

base_path = os.path.join(path, 'Food Classification dataset')

files_list = os.listdir(base_path)
files_list = os.listdir(base_path)[:15]

data_files = []

for i in range(len(files_list)):
    class_path = os.path.join(base_path, files_list[i])
    path_obj = os.listdir(class_path)
    path_obj_full = [os.path.join(class_path, f) for f in path_obj]
    data_files.append(path_obj_full)

In [7]:
gran_list = []
for i in range(len(data_files)):
    gran_list.append(int(len(data_files[i])))
min(gran_list)

144

In [8]:
from PIL import Image
import random

data_x = []
data_y = []
size = (260, 260)

target_count = 2 * min(gran_list)

for class_idx, file_list in enumerate(data_files):
    num_samples = len(file_list)
    
    for j in range(target_count):

        img_idx = j % num_samples
        img = Image.open(file_list[img_idx])
        res = img.convert('RGB').resize(size)
        res = np.array(res) / 255.0
        
        aug_choice = random.choice(['none', 'rotate', 'flip', 'brightness', 'contrast'])
        
        if aug_choice == 'rotate':
            k = random.randint(1, 3)
            res = np.rot90(res, k)
        elif aug_choice == 'flip':
            if random.choice([True, False]):
                res = np.fliplr(res)
            else:
                res = np.flipud(res)
        elif aug_choice == 'brightness':
            res = res * random.uniform(0.7, 1.3)
            res = np.clip(res, 0, 1)
        elif aug_choice == 'contrast':
            mean = np.mean(res, axis=(0, 1), keepdims=True)
            res = (res - mean) * random.uniform(0.7, 1.3) + mean
            res = np.clip(res, 0, 1)
        
        data_x.append(res)
        data_y.append(class_idx)

In [9]:
from sklearn.model_selection import train_test_split

data_x = np.array(data_x)
data_y = np.array(data_y)

x_train, x_test, y_train, y_test = train_test_split(
    data_x, 
    data_y, 
    test_size=0.15, 
    random_state=42
)

In [10]:
del data_x
del data_y

In [11]:
import keras
from tensorflow.keras.callbacks import EarlyStopping
from keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras import layers

In [13]:
model = keras.Sequential([
    keras.Input(shape = (260, 260, 3)),
    keras.layers.Conv2D(5, (3, 3), padding = 'same', strides = 2),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(5, (3, 3), padding = 'same', activation ='relu', strides = 2),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(32, activation ='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation ='relu'),
    keras.layers.Dense(15, activation ='softmax')
])

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-3),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

early_stopping = EarlyStopping(
                    monitor='val_loss',
                    mode='min',
                    patience = 50,
                    min_delta = 0.01,
                    verbose = 0,
                    restore_best_weights = True
                    )

In [14]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 32,
        epochs = 2000,
        verbose = 0,
        callbacks = [early_stopping], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 32
        )

2026-03-02 22:55:38.438620: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2680204800 exceeds 10% of free system memory.
2026-03-02 22:55:40.931389: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2680204800 exceeds 10% of free system memory.
2026-03-02 22:55:44.600131: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2026-03-02 22:55:50.137134: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7653dc8cd2c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-02 22:55:50.137155: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 SUPER, Compute Capability 8.9
2026-03-02 22:55:50.240376: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-02 22:55:50.515493: I ./tensorflow/compiler/jit/devic

In [15]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля нейронной сети точность для обучающей \ тестовой выборки:', keras_train_acc, ' \ ', keras_test_acc)

2026-03-02 22:56:44.101763: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2978726400 exceeds 10% of free system memory.
2026-03-02 22:56:46.766684: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 2978726400 exceeds 10% of free system memory.


21/21 [==============================] - 0s 7ms/step - loss: 2.0824 - accuracy: 0.3318

для нейронной сети точность для обучающей \ тестовой выборки: 0.5302287340164185  \  0.33179011940956116


In [24]:
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [28]:
base_model = EfficientNetB2(
    include_top=False, 
    weights='imagenet',
    input_shape=(260, 260, 3),
    pooling='max'
)
base_model.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.3)(x)
x = BatchNormalization()(x)
x = Dense(32, activation='relu')(x)
predictions = Dense(15, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(  optimizer = keras.optimizers.Adam(learning_rate = 1e-3),
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
                )

# model.summary()

In [29]:
model.fit(
        x = x_train,
        y = y_train,
        batch_size = 32,
        epochs = 2000,
        verbose = 1,
        callbacks = [early_stopping], 
        shuffle = True,
        validation_split = 0.1,
        validation_data = None,
        validation_batch_size = 32
        )

Epoch 1/2000
104/104 [==============================] - 12s 72ms/step - loss: 2.8037 - accuracy: 0.0636 - val_loss: 2.7418 - val_accuracy: 0.0707
Epoch 2/2000
104/104 [==============================] - 6s 56ms/step - loss: 2.7568 - accuracy: 0.0693 - val_loss: 2.7469 - val_accuracy: 0.0598
Epoch 3/2000
104/104 [==============================] - 6s 56ms/step - loss: 2.7376 - accuracy: 0.0654 - val_loss: 2.7138 - val_accuracy: 0.0571
Epoch 4/2000
104/104 [==============================] - 6s 55ms/step - loss: 2.7219 - accuracy: 0.0663 - val_loss: 2.7084 - val_accuracy: 0.0761
Epoch 5/2000
104/104 [==============================] - 6s 55ms/step - loss: 2.7198 - accuracy: 0.0690 - val_loss: 2.7103 - val_accuracy: 0.0788
Epoch 6/2000
104/104 [==============================] - 6s 55ms/step - loss: 2.7209 - accuracy: 0.0623 - val_loss: 2.7087 - val_accuracy: 0.0788
Epoch 7/2000
104/104 [==============================] - 6s 56ms/step - loss: 2.7126 - accuracy: 0.0748 - val_loss: 2.7092 - val_a

In [ ]:
train_loss, keras_train_acc = model.evaluate(x_train, y_train)
test_loss, keras_test_acc = model.evaluate(x_test, y_test)
print('\nдля нейронной сети точность для обучающей \ тестовой выборки:', keras_train_acc, ' \ ', keras_test_acc)